# PSC walkthrough: from the bulk file to per-company features

This notebook walks a new user from the Companies House **PSC** (persons with significant control)
bulk file to usable tables and per-company features, and shows what the results look like. It is a
broad tour, not a reference — it calls the project's own code at every step and never reimplements
any of it.

## 1. What the PSC data is

The **PSC register** records who ultimately owns or controls each UK company — the people (or other
companies) with significant control, and, where no such person is identified, the *statement* the
company filed instead (for example "the company knows of no registrable person"). Companies House
publishes the whole register once a day as a **bulk snapshot**: 32 files of newline-delimited JSON
("NDJSON" — one JSON object per line), together about 16 million records.

Two things make it awkward. First, it is **personal data**: individual PSCs have names, dates of
birth, nationalities and addresses. This project therefore publishes *code only* for PSC, and this
notebook runs in **governed mode**, which drops every personal field and keeps only a pseudonymous
key. Second, it is a **snapshot, not an event log**: you get today's picture, and corrections over
time are lost unless you archive each day.

For the dataset page and the deeper guide, see
[`docs/site/datasets/psc.qmd`](site/datasets/psc.qmd) and
[`docs/psc-data-guide.md`](psc-data-guide.md). The feature table this notebook builds is documented
in [`docs/psc-bulk-features.md`](psc-bulk-features.md).

## 2. Setup

One switch, `SAMPLE`, controls the whole notebook:

- `SAMPLE = True` (default) loads **one of the 32 parts** — seconds to a couple of minutes on a
  laptop. **Every number below is then from a partial sample** and is labelled as such.
- `SAMPLE = False` loads the **full snapshot**. We show the measured time and memory before the
  heavy cell so you can decide.

We also read the HMAC secret from the environment and assert governed mode. The next cell never
prints the secret's value — only whether it is set.

In [1]:
import os
import glob
import json
import time
from pathlib import Path

from ukcompany.cli import load_dotenv

# The one switch.
SAMPLE = True

# Governed only: no personal field is ever loaded or shown in this notebook.
DATA_GOVERNANCE = True
assert DATA_GOVERNANCE, "this notebook is governed-only by design"

SNAPSHOT_DATE = "2026-09-25"
# Where the extracted NDJSON parts live (adapt to your machine). A "part" is one of the 32
# psc-snapshot-<date>_Nof32.txt files.
PARTS_DIR = Path.home() / "Downloads" / "psc" / SNAPSHOT_DATE
# The committed manifest (sha256 per part) produced by `ukcompany-psc fetch`.
MANIFEST = Path("data/psc") / SNAPSHOT_DATE / "manifest.json"

tag = "sample" if SAMPLE else "full"
LOAD_DIR = Path("data/psc") / f"{SNAPSHOT_DATE}-{tag}" / "governed"
FEAT_DIR = Path("data/psc") / f"{SNAPSHOT_DATE}-{tag}" / "features-governed"

# The HMAC secret is read from the environment / .env — never written into the notebook.
load_dotenv()
secret = os.environ.get("PSC_PERSON_KEY_SECRET")
print("SAMPLE          :", SAMPLE)
print("data_governance :", DATA_GOVERNANCE)
print("snapshot date   :", SNAPSHOT_DATE)
print("PSC_PERSON_KEY_SECRET is set:", bool(secret))   # value is never shown

SAMPLE          : True
data_governance : True
snapshot date   : 2026-09-25
PSC_PERSON_KEY_SECRET is set: True


**Why a secret in governed mode?** Governed mode drops the raw personal fields (name, date of
birth, nationality, address) but keeps one derived column: `person_key`, an **HMAC** — a keyed,
one-way fingerprint — of the person's baseline identity. It lets the ungoverned tier count how many
companies a person controls *without* storing who they are. Because that key can only be computed
with the secret, the loader **requires `PSC_PERSON_KEY_SECRET` even in governed mode** and raises if
it is missing (see `ukcompany/psc/loader.py`: `if data_governance and not person_key_secret: raise`).
Set it in your shell or a gitignored `.env`; never commit a value.

## 3. Get the snapshot

The repeatable way to fetch a fresh snapshot is `ukcompany-psc refresh`, which downloads the 32
parts, **verifies their checksums** against the manifest, prunes old archives, extracts, loads, and
reconciles the totals — failing loudly if anything doesn't add up. Here we point at an archive that
is already on disk and run just the checksum verification, so the walkthrough stays fast.

In [2]:
from ukcompany.psc.archive import verify_manifest

if SAMPLE:
    parts_glob = str(PARTS_DIR / f"psc-snapshot-{SNAPSHOT_DATE}_1of32.txt")   # one part
else:
    parts_glob = str(PARTS_DIR / f"psc-snapshot-{SNAPSHOT_DATE}_*of32.txt")   # all 32

n_parts = len(glob.glob(parts_glob))
print(("SAMPLE: using 1 of 32 parts" if SAMPLE else "FULL: using all 32 parts"),
      f"-> {n_parts} file(s) matched")

# Checksum verification is over the whole archive (the manifest covers all 32 zips), so it is the
# same integrity check whichever mode we are in. Empty result == every part matches its sha256.
problems = verify_manifest(MANIFEST, PARTS_DIR)
print("checksum problems:", problems if problems else "none — archive intact")

SAMPLE: using 1 of 32 parts -> 1 file(s) matched


checksum problems: none — archive intact


## 4. Load into tables

`load_psc` parses the NDJSON into two Parquet tables and writes a load report:

- **`psc_records.parquet`** — one row per record (each individual, corporate or legal-person PSC,
  each statement, each exemption).
- **`psc_noc.parquet`** — one row per *nature-of-control* assertion (a single record can hold
  several, e.g. "owns 75–100% of shares" **and** "holds 75–100% of voting rights").

The load report includes the **totals reconciliation** — our parsed counts checked against the
authoritative `totals#…` line the snapshot ships — and the **bad-lines split**. In `SAMPLE` mode a
single part has no totals line, so that reconciliation is `None`; it only runs on the full snapshot.

> **Full-snapshot cost.** The next cell is cheap in `SAMPLE` mode. With `SAMPLE = False` the last
> measured full load was about **11 minutes and ~10 GB of memory** (8 GB DuckDB limit plus
> overhead). Set `SAMPLE = False` only when you want the whole register.

In [3]:
from ukcompany.psc.loader import load_psc

t0 = time.time()
report = load_psc(
    parts_glob, LOAD_DIR, SNAPSHOT_DATE,
    data_governance=DATA_GOVERNANCE, person_key_secret=secret, memory_limit_gb=8,
)
print(f"loaded in {time.time() - t0:.0f}s   (SAMPLE={SAMPLE})")
print("lines:", f"{report['n_lines']:,}", " bad lines:", report["n_bad_lines"])
print("categories sum to lines:", report["categories_sum_to_lines"])

loaded in 20s   (SAMPLE=True)
lines: 500,000  bad lines: 0
categories sum to lines: True


In [4]:
import duckdb

for name in ("psc_records", "psc_noc"):
    path = LOAD_DIR / f"{name}.parquet"
    n = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
    cols = duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{path}')").fetchall()
    print(f"\n{name}.parquet — {n:,} rows, {len(cols)} columns")
    print("  columns:", ", ".join(c[0] for c in cols))


psc_records.parquet — 500,000 rows, 39 columns
  columns: snapshot_date, part, line_no, company_number, company_number_from_links, company_number_mismatch, psc_id, record_id, kind, category, country_of_residence, reg_number_raw, reg_number_norm, country_registered, legal_form, legal_authority, place_registered, statement, is_sanctioned, iv_identity_verified_on_raw, iv_verification_start_on_raw, iv_verification_end_on_raw, iv_verification_statement_date_raw, iv_verification_statement_due_on_raw, iv_acsp_name, iv_aml_supervisory_bodies, notified_on_raw, ceased_on_raw, notified_on, ceased_on, ceased, f_ceased_before_notified, f_pre_regime, f_ceased_out_of_range, f_age_under_16_at_notified, birth_band_5y, postcode_district, person_key, person_key_strict

psc_noc.parquet — 1,064,162 rows, 6 columns
  columns: record_id, right_raw, base_right, suffix_family, band, core


In [5]:
# Load report: records by kind, the totals reconciliation, and the bad-lines split.
print("records by kind (category_counts):")
for k, v in sorted(report["category_counts"].items(), key=lambda kv: -kv[1]):
    print(f"  {k:<14} {v:,}")
print("\ntotals reconciliation:", report.get("totals_reconciliation"),
      "" if report.get("totals_reconciliation") else "(no totals line in a single part — SAMPLE)")
print("bad-line reasons:", report["bad_line_reasons"])

records by kind (category_counts):
  individual     461,707
  corporate      37,528
  legal_person   651
  bo_individual  70
  super_secure   29
  bo_corporate   13
  bo_legal       2

totals reconciliation: None (no totals line in a single part — SAMPLE)
bad-line reasons: {'invalid_json': 0, 'missing_kind': 0}


## 5. What a record looks like

To show the shape of a record without displaying anyone's real data, we use a **synthetic** record
(invented person, same structure as the project's test fixtures). The interesting part is the
`natures_of_control` list. Each entry is a long code like
`ownership-of-shares-75-to-100-percent-as-trust`; the shared `psc_natures` decomposition splits it
into three pieces:

- the **core right** — what kind of control it is (here, ownership of shares);
- the **band** — how much (here, 75–100%);
- the **suffix family** — on whose behalf (here, "as trust").

In [6]:
from ukcompany.psc_natures import decompose_nature, classify_kind, summarise_natures

# Synthetic record — invented data, never from the real file.
synthetic = {
    "kind": "individual-person-with-significant-control",
    "name_elements": {"forename": "Alex", "surname": "Example"},  # shown only to prove it's fake
    "natures_of_control": [
        "ownership-of-shares-75-to-100-percent-as-trust",
        "voting-rights-75-to-100-percent",
        "right-to-appoint-and-remove-directors",
    ],
}

print("coarse kind:", classify_kind(synthetic["kind"]))
print()
for raw in synthetic["natures_of_control"]:
    p = decompose_nature(raw)
    print(f"{raw}\n    core={p.core!r}  band={p.band!r}  suffix={p.suffix_family!r}\n")

# summarise_natures rolls a set of codes into the per-company signals used later:
print("summary:", summarise_natures(synthetic["natures_of_control"]))

coarse kind: individual

ownership-of-shares-75-to-100-percent-as-trust
    core='ownership-of-shares'  band='75-to-100-percent'  suffix='as-trust'

voting-rights-75-to-100-percent
    core='voting-rights'  band='75-to-100-percent'  suffix='plain'

right-to-appoint-and-remove-directors
    core='right-to-appoint-and-remove'  band=None  suffix='plain'

summary: NatureSummary(max_ownership_band='75-to-100-percent', max_voting_band='75-to-100-percent', has_appointment_rights=True, has_significant_influence=False, n_distinct_natures=3, unmapped=())


## 6. Build per-company features

`build_psc_features` turns the per-record tables into **one row per company**, reusing exactly the
`psc_natures` definitions above (no second implementation). In governed mode it writes the feature
table **without** the two companies-per-person band columns (those are person-linkage and live only
in the ungoverned tier). Every column is registered in `FIELD_DOCS`, shown below.

In [7]:
from ukcompany.psc.features import build_psc_features, FEATURE_COLUMNS, GOVERNED_FEATURE_DROPPED
from ukcompany.derive import FIELD_DOCS

t0 = time.time()
freport = build_psc_features(
    LOAD_DIR / "psc_records.parquet", LOAD_DIR / "psc_noc.parquet", FEAT_DIR, SNAPSHOT_DATE,
    data_governance=DATA_GOVERNANCE, memory_limit_gb=8,
)
print(f"built features in {time.time() - t0:.0f}s   companies: {freport['n_companies']:,}"
      f"   (SAMPLE={SAMPLE})")
print("governed tier drops (person-linkage):", GOVERNED_FEATURE_DROPPED)

built features in 3s   companies: 406,731   (SAMPLE=True)
governed tier drops (person-linkage): ('companies_per_person_band_ever', 'companies_per_person_band_active')


In [8]:
# Each governed feature column with its FIELD_DOCS description.
docs_by_field = {d["field"]: d for d in FIELD_DOCS}
for col in FEATURE_COLUMNS:
    d = docs_by_field.get(col)
    if col == "company_number":
        print(f"{col:<32} the Companies House number (join key)")
    elif d:
        print(f"{col:<32} {d['definition']}")
    else:
        print(f"{col:<32} (not in FIELD_DOCS)")

company_number                   the Companies House number (join key)
psc_n_records                    Count of PSC records on the list resource (active + ceased).
psc_n_ceased                     PSC records carrying a ceased_on date.
psc_natures_of_control           Sorted, distinct, comma-joined natures-of-control across ACTIVE PSC records (e.g. ownership-of-shares-25-to-50-percent).
psc_max_ownership_band           Highest share-ownership band held by any active PSC: one of 25-to-50-percent / 50-to-75-percent / 75-to-100-percent.
psc_max_voting_band              Highest voting-rights band held by any active PSC (same band values as psc_max_ownership_band).
psc_has_appointment_rights       True when any active PSC holds a right-to-appoint-and-remove nature.
psc_has_significant_influence    True when any active PSC holds significant-influence-or-control.
psc_n_distinct_natures           Count of distinct verbatim nature-of-control codes across active PSCs.
psc_n_individual          

## 7. Broad results

Everything here is **aggregates only** — no company numbers, no rows. Counts below 10 are shown as
`<10`, the same suppression the site uses (the site's helper `psc_site_summaries.disp` lives on an
unmerged branch, so we apply the `<10` rule inline here — noted as a gap). **In `SAMPLE` mode these
are from one of 32 parts** — read them as shape, not totals.

In [9]:
import duckdb

FEAT = str(FEAT_DIR / "psc_company_features.parquet")
SUPPRESS = 10   # site convention: counts below this are shown as "<10"

def disp(n):
    return "<10" if (n is not None and 0 < n < SUPPRESS) else (f"{n:,}" if n is not None else "")

label = "PARTIAL SAMPLE (1/32 parts)" if SAMPLE else "FULL SNAPSHOT"
print("results basis:", label)

results basis: PARTIAL SAMPLE (1/32 parts)


**Records by kind** — the mix of individuals, corporates, statements and so on.

In [10]:
for k, v in sorted(report["category_counts"].items(), key=lambda kv: -kv[1]):
    print(f"  {k:<14} {disp(v)}")

  individual     461,707
  corporate      37,528
  legal_person   651
  bo_individual  70
  super_secure   29
  bo_corporate   13
  bo_legal       <10


**Active vs ceased** — a ceased PSC has stepped down but stays in the file for history.

In [11]:
row = duckdb.sql(
    f"SELECT SUM(psc_n_records) r, SUM(psc_n_ceased) c FROM read_parquet('{FEAT}')"
).fetchone()
total, ceased = row[0] or 0, row[1] or 0
print(f"  active  {disp(total - ceased)}")
print(f"  ceased  {disp(ceased)}")

  active  404,938
  ceased  95,062


**Information state per company** — `identified` (at least one live PSC), `statement_only` (no PSC
but a filed statement) or `none_reported`.

In [12]:
rows = duckdb.sql(
    f"SELECT psc_information_state, COUNT(*) n FROM read_parquet('{FEAT}') "
    "GROUP BY 1 ORDER BY n DESC"
).fetchall()
for state, n in rows:
    print(f"  {state:<16} {disp(n)}")

  identified       347,310
  none_reported    59,421


**Nature-of-control cores** — what kind of control is asserted, decomposed with `psc_natures`
(reusing `decompose_nature`, not re-counting raw strings).

In [13]:
from collections import Counter
cores = Counter()
rows = duckdb.sql(f"SELECT DISTINCT right_raw FROM read_parquet('{LOAD_DIR / 'psc_noc.parquet'}')").fetchall()
# map each distinct code -> core via the shared function, then weight by occurrences
counts = dict(duckdb.sql(
    f"SELECT right_raw, COUNT(*) FROM read_parquet('{LOAD_DIR / 'psc_noc.parquet'}') GROUP BY 1"
).fetchall())
for code_str, n in counts.items():
    core = decompose_nature(code_str).core or "(unmapped)"
    cores[core] += n
for core, n in cores.most_common():
    print(f"  {core:<34} {disp(n)}")

  ownership-of-shares                451,605
  voting-rights                      320,032
  right-to-appoint-and-remove        216,821
  significant-influence-or-control   71,007
  right-to-share-surplus-assets      4,184
  part-right-to-share-surplus-assets 508
  registered-owner-as-nominee        <10


**Maximum ownership band per company** — the largest share-ownership band each company reports.

In [14]:
rows = duckdb.sql(
    f"SELECT COALESCE(psc_max_ownership_band, '(none)') b, COUNT(*) n "
    f"FROM read_parquet('{FEAT}') GROUP BY 1 ORDER BY n DESC"
).fetchall()
for band, n in rows:
    print(f"  {band:<22} {disp(n)}")

  75-to-100-percent      214,927
  (none)                 94,824
  25-to-50-percent       78,353
  50-to-75-percent       18,627


**ECCTA identity-verification counts** — how many PSC records are inside the new identity-checking
regime. **Point-in-time:** the rollout is ongoing, and `n_psc_id_statement_filed` / `_due` reflect
the *current* verification cycle (they can fall when it resets) — they are not lifetime totals.

In [15]:
row = duckdb.sql(
    f"SELECT SUM(n_psc_id_verified), SUM(n_psc_id_verification_due), "
    f"SUM(n_psc_id_statement_filed) FROM read_parquet('{FEAT}')"
).fetchone()
for label_, v in zip(("verified (block present)", "verification due", "statement filed"), row):
    print(f"  {label_:<26} {disp(v or 0)}")

  verified (block present)   194,135
  verification due           78,780
  statement filed            78,780


**Top statement codes** — what companies filed when no PSC is identified (suppressed `<10`).

In [16]:
codes = Counter()
for (s,) in duckdb.sql(
    f"SELECT active_psc_statement_codes FROM read_parquet('{FEAT}') WHERE active_psc_statement_codes IS NOT NULL"
).fetchall():
    for c in s.split(","):
        codes[c] += 1
for code_str, n in codes.most_common(10):
    print(f"  {code_str:<48} {disp(n)}")

## 8. Join to the register features

Coverage asks: of the live companies on the register, how many have a PSC record in the snapshot?
This only makes sense on the **full** snapshot, so in `SAMPLE` mode we show the committed
full-snapshot result (`scripts/psc_coverage_check.py`, written to
[`docs/psc-coverage-2026-09-25.md`](psc-coverage-2026-09-25.md)) rather than a misleading 1/32
number. With `SAMPLE = False` you can re-run the script yourself against the full feature table.

The key point is the **"outside the PSC regime"** explanation: some company types (Charitable
Incorporated Organisations, Registered Societies, Royal Charter companies, …) have no PSCs to file,
so their ~100% "uncovered" is expected, not a gap — while private limited companies are ~99%
covered.

In [17]:
cov = Path("docs/psc-coverage-2026-09-25.md")
if SAMPLE:
    # Show the committed full-snapshot coverage summary (first lines + the type table head).
    text = cov.read_text().splitlines()
    print("\n".join(text[:22]))
else:
    import subprocess
    out = subprocess.run(
        ["python", "scripts/psc_coverage_check.py",
         "--features", str(FEAT_DIR / "psc_company_features.parquet"),
         "--register", str(Path.home() / "Downloads" / "BasicCompanyDataAsOneFile-2026-09-01.csv"),
         "--register-date", "2026-09-01", "--snapshot-date", SNAPSHOT_DATE,
         "--out", "docs/psc-coverage-notebook.md"],
        capture_output=True, text=True,
    )
    print(out.stdout[-800:] or out.stderr[-800:])

# PSC bulk-snapshot coverage vs the register (2026-09-25 snapshot)

PSC snapshot: **2026-09-25**. Register: Basic Company Data one-file, **2026-09-01**. Join key: company number. Aggregates only.

- Register companies: **5,689,366**
- Feature-table companies (any PSC record in the snapshot): **10,932,116**
- Register companies **covered** (have a PSC row): **5,536,892** (**97.32%**)
- Register companies **uncovered**: **152,474**
- Register **Active**-status companies covered: **5,049,044 / 5,171,599** (**97.63%**)
- Snapshot companies **absent from the register** (dissolved / historical): **5,395,224**

## Uncovered register companies by company type

| CompanyCategory | register | covered | uncovered | uncovered % |
|---|---:|---:|---:|---:|
| Charitable Incorporated Organisation | 40,416 | 0 | 40,416 | 100.0% |
| Limited Partnership | 61,003 | 20,637 | 40,366 | 66.2% |
| Private Limited Company | 5,261,978 | 5,227,892 | 34,086 | 0.6% |
| Other company type | 15,508 | 32 | 15,476 | 9

## 9. What you can't do with this

Governed PSC data is deliberately limited:

- **No person identity.** Names, dates of birth and addresses are dropped; only the pseudonymous
  `person_key` remains, and it is never shown. You cannot recover who a person is from this table.
- **No nationality.** It is removed entirely and never summarised.
- **Companies-per-person is ungoverned-only and banded.** How many companies one person controls is
  person-linkage; it exists only in the ungoverned tier, only as coarse bands (`1` / `2–10` / `11+`),
  and never appears here. On a single-PSC company even a band can point at an identifiable person,
  which is why it is withheld in governed mode.
- **`notified_on` is not verified.** The date a PSC was notified is as-filed and not independently
  checked — treat it as a claim, not a fact.

These limits are the point, not a shortfall: the register is personal data under the Open Government
Licence's personal-data carve-out. See the governance sections of
[`docs/psc-data-guide.md`](psc-data-guide.md) and [`docs/psc-bulk-features.md`](psc-bulk-features.md).

*Run basis: everything above with `SAMPLE = True` is from one of the 32 parts — a partial sample,
shown for shape. Set `SAMPLE = False` for the full register (≈11 min, ≈10 GB).*